<a href="https://colab.research.google.com/github/JaronLouise/ARQUILLO-HCI-FINALPROJECT/blob/main/MODEL_TRAINING_STAGE1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Cell 0 — Google Colab Setup
Run **Cell 0** first (mounts Google Drive, verifies packages).  
Then set `DATA_ROOT_OVERRIDE` in **Cell 2** to your dataset path on Google Drive,  
e.g. `/content/drive/MyDrive/ferning_project/Preprocessed-...`

> **Tip:** Enable GPU runtime first — *Runtime → Change runtime type → T4 GPU*


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  GOOGLE COLAB SETUP — Run this cell FIRST, then proceed to Cell 1 ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys

# Install any packages not pre-installed on Colab
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-python-headless", "scipy", "scikit-learn"])

# Mount Google Drive (outputs + dataset will live here)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

print("[OK] Drive mounted and packages verified.")
print("     --> Set DATA_ROOT_OVERRIDE in Cell 2 to your dataset path on Drive.")


Mounted at /content/drive
[OK] Drive mounted and packages verified.
     --> Set DATA_ROOT_OVERRIDE in Cell 2 to your dataset path on Drive.


# Stage 1 — Binary Ferning Detector
### LOSO Cross-Validation Training Pipeline
**Chapter 3 compliant** | Designed to hand off cleanly to Stage 2, PINN, and Evaluation notebooks.

**Run cells top to bottom in order.**

---
#### Notebook Outputs (used by downstream notebooks)
| File | Used By |
|---|---|
| `outputs/stage1/fold_splits.csv` | Stage 2, PINN |
| `outputs/stage1/master_patch_index.csv` | Stage 2 |
| `outputs/stage1/all_results.csv` | Evaluation |
| `outputs/stage1/best_model_name.txt` | PINN |
| `outputs/stage1/{model}/fold{n}/best_model.keras` | PINN, Evaluation (Grad-CAM) |
| `outputs/stage1/{model}/fold{n}/val_predictions.npz` | Evaluation |
| `outputs/stage1/{model}/fold{n}/history.csv` | Evaluation |


---
## Cell 1 — Imports & Environment Setup

In [5]:

import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
from scipy import stats
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB3, DenseNet121, ResNet50
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Fix Windows console encoding
if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

print(f"Python     : {sys.version.split()[0]}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")

physical_gpus = tf.config.list_physical_devices("GPU")
if physical_gpus:
    for gpu in physical_gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print(f"  Could not set memory growth: {e}")
    print(f"Device     : GPU — {[g.name for g in physical_gpus]}")
else:
    print("Device     : CPU (no GPU detected — training will be slow)")


Python     : 3.12.12
TensorFlow : 2.19.0
Keras      : 3.10.0
Device     : CPU (no GPU detected — training will be slow)



---
## Cell 2 — Configuration
All hyperparameters in one place. Edit `DATA_ROOT_OVERRIDE` to point to your preprocessed patches folder.

**All values match Chapter 3 of the thesis manuscript exactly.**


In [4]:

# ── Path override ─────────────────────────────────────────────────────────
# Run the snippet below in a separate cell if you are unsure of your path:
#   import subprocess
#   r = subprocess.run(['find', '/content/drive/MyDrive', '-maxdepth', '4',
#                        '-name', 'Preprocessed', '-type', 'd'], capture_output=True, text=True)
#   print(r.stdout or 'Not found')
#
# Set DATA_ROOT_OVERRIDE to the folder that CONTAINS your Preprocessed/ directory.
# It must be a filesystem path like /content/drive/MyDrive/...
# Do NOT paste a Google Drive share URL here.
DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/ferning_project"

# ── Dataset & class configuration (§3.2) ──────────────────────────────────
KNOWN_CLASSES        = ["PF", "CF", "NF"]   # PF=Pure Ferning, CF=Contaminated, NF=No Ferning
FERNING_CLASSES      = ["PF", "CF"]          # Both count as Stage 1 positive
CLASS_NAMES          = ["No Ferning", "Ferning"]
POSITIVE_CLASS_INDEX = 1
NUM_CLASSES          = 2                      # Stage 1 is binary

# ── Input shape (§3.5 — 128×128 RGB patches) ──────────────────────────────
INPUT_SHAPE = (128, 128, 3)

# ── Training hyperparameters (§3.5) ───────────────────────────────────────
EPOCHS_STAGE_A  = 10     # Head-only warmup (§3.2 Transfer Learning Stage A)
EPOCHS_STAGE_B  = 20     # Top-30% unfrozen (§3.2 Transfer Learning Stage B)
EPOCHS_STAGE_C  = 50     # Full fine-tune, early-stopped (§3.2 Transfer Learning Stage C)
LR_STAGE_A      = 1e-4   # Initial learning rate (§3.5)
LR_STAGE_B      = 1e-5   # Reduced for domain adaptation (§3.2)
LR_STAGE_C      = 1e-6   # Further reduced for full fine-tune (§3.2)
BATCH_SIZE      = 32     # (§3.5)
DROPOUT_RATE    = 0.5    # Applied before final Dense layer (§3.2)
EARLY_STOP_PAT  = 10     # Patience for EarlyStopping on val_loss (§3.2)
LR_REDUCE_PAT   = 5      # Patience for ReduceLROnPlateau
LR_REDUCE_FAC   = 0.5    # Factor for ReduceLROnPlateau
LR_MIN          = 1e-7   # Minimum learning rate floor
RANDOM_SEED     = 42     # Fixed seed for full reproducibility (§3.10.4)

# ── Class-aware augmentation multipliers (§3.5, Table 6) ──────────────────
AUG_MULTIPLIER = {"PF": 1.0, "CF": 1.8, "NF": 1.0}

# ── Decision threshold for Stage 1 (§3.6) ─────────────────────────────────
THRESHOLD = 0.5

# ── Architectures to train (§3.2) ─────────────────────────────────────────
MODEL_NAMES = ["EfficientNetB3", "DenseNet121", "ResNet50", "SwinTransformerTiny"]

# ── Validate DATA_ROOT_OVERRIDE is a filesystem path, not a URL ───────────
if DATA_ROOT_OVERRIDE.startswith("http"):
    raise ValueError(
        "DATA_ROOT_OVERRIDE must be a filesystem path, not a URL.\n"
        "Open your Google Drive at drive.google.com, right-click your dataset\n"
        "folder, choose 'Organise > Move' to note the folder name, then set:\n"
        "  DATA_ROOT_OVERRIDE = \"/content/drive/MyDrive/<your-folder-name>\""
    )

# ── Resolve paths ──────────────────────────────────────────────────────────
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

DATA_ROOT   = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else NOTEBOOK_DIR / "data"
DATASET_DIR = (DATA_ROOT / "Preprocessed") if (DATA_ROOT / "Preprocessed").exists() else DATA_ROOT

# Outputs saved to Drive so they persist after the Colab session ends
OUTPUT_ROOT        = Path("/content/drive/MyDrive/ferning_project/outputs")
STAGE1_OUTPUT_DIR = OUTPUT_ROOT / "stage1"
STAGE1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_INDEX_PATH = STAGE1_OUTPUT_DIR / "master_patch_index.csv"
FOLD_SPLITS_PATH  = STAGE1_OUTPUT_DIR / "fold_splits.csv"

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"Data root    : {DATA_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}  {'[EXISTS]' if DATASET_DIR.exists() else '[NOT FOUND — check DATA_ROOT_OVERRIDE]'}")
print(f"Stage1 out   : {STAGE1_OUTPUT_DIR}")
print(f"Models       : {MODEL_NAMES}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Random seed  : {RANDOM_SEED}")

if not DATASET_DIR.exists():
    print()
    print("[HINT] Dataset not found. To locate your folder on Drive, run this in a new cell:")
    print("  import subprocess")
    print("  r = subprocess.run(['find', '/content/drive/MyDrive', '-maxdepth', '5',")
    print("                       '-name', 'Preprocessed', '-type', 'd'],")
    print("                      capture_output=True, text=True)")
    print("  print(r.stdout or 'Not found')")


Data root    : /content/drive/MyDrive/ferning_project
Dataset dir  : /content/drive/MyDrive/ferning_project/Preprocessed  [EXISTS]
Stage1 out   : /content/drive/MyDrive/ferning_project/outputs/stage1
Models       : ['EfficientNetB3', 'DenseNet121', 'ResNet50', 'SwinTransformerTiny']
Batch size   : 32
Random seed  : 42



---
## Cell 3 — Build Master Patch Index
Scans the preprocessed dataset and builds `master_patch_index.csv`.
Supports Layout A (`PF/`, `CF/`, `NF/` subfolders) and Layout B (`SampleX_PF/` subfolders).

Set `FORCE_REGENERATE = True` to rebuild if your dataset changes.


In [6]:

FORCE_REGENERATE = True  # Set False once index is built

def generate_master_index(dataset_dir: Path, output_path: Path) -> pd.DataFrame:
    """
    Walk dataset_dir and build master_patch_index.csv.
    Each row represents one .npy patch with its sample_id, class, and Stage 1 label.
    """
    if not dataset_dir.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {dataset_dir}\n"
            f"Check DATA_ROOT_OVERRIDE in Cell 2."
        )

    records = []

    # Layout A: class subfolders PF/, CF/, NF/
    direct_class_dirs = [dataset_dir / c for c in KNOWN_CLASSES if (dataset_dir / c).exists()]
    # Layout B: sample subfolders SampleX_PF/, SampleX_CF/, etc.
    sample_dirs = [
        d for d in sorted(dataset_dir.iterdir())
        if d.is_dir() and any(d.name.upper().endswith(f"_{c}") for c in KNOWN_CLASSES)
    ]

    if direct_class_dirs:
        print("  Detected Layout A: class subfolders (PF/, CF/, NF/)")
        for class_name in KNOWN_CLASSES:
            class_dir = dataset_dir / class_name
            if not class_dir.exists():
                continue
            npy_files = sorted(class_dir.glob("*.npy"))
            print(f"  {class_name}: {len(npy_files):,} patches")
            for fp in npy_files:
                stem = fp.stem
                parts = stem.rsplit("_", 1)
                sample_id = parts[0] if len(parts) > 1 else stem
                records.append({
                    "sample_id"   : sample_id,
                    "patch_id"    : stem,
                    "class"       : class_name,
                    "label_stage1": 1 if class_name in FERNING_CLASSES else 0,
                    "path"        : str(fp.resolve()),
                })

    elif sample_dirs:
        print(f"  Detected Layout B: {len(sample_dirs)} sample folders")
        for sample_dir in sample_dirs:
            folder_name = sample_dir.name
            class_name  = folder_name.rsplit("_", 1)[-1].upper()
            if class_name not in KNOWN_CLASSES:
                continue
            npy_files = sorted(sample_dir.glob("npy/*.npy"))
            sample_id = folder_name
            print(f"  {folder_name}: {len(npy_files):,} patches (class={class_name})")
            for fp in npy_files:
                records.append({
                    "sample_id"   : sample_id,
                    "patch_id"    : fp.stem,
                    "class"       : class_name,
                    "label_stage1": 1 if class_name in FERNING_CLASSES else 0,
                    "path"        : str(fp.resolve()),
                })
    else:
        raise RuntimeError(
            f"No recognisable class folders found in {dataset_dir}.\n"
            f"Expected PF/CF/NF subfolders or SampleX_PF/CF/NF subfolders."
        )

    if not records:
        raise RuntimeError("No .npy files found. Check your dataset folder structure.")

    df = pd.DataFrame(records)
    df.to_csv(output_path, index=False)
    print(f"\n  Saved -> {output_path}")
    return df


print("=" * 60)
print("GENERATING master_patch_index.csv")
print("=" * 60)

if MASTER_INDEX_PATH.exists() and not FORCE_REGENERATE:
    print("  [SKIP] Already exists — loading from disk.")
    master_index = pd.read_csv(MASTER_INDEX_PATH)
else:
    master_index = generate_master_index(DATASET_DIR, MASTER_INDEX_PATH)

print(f"\n  Total patches  : {len(master_index):,}")
print(f"  Unique samples : {master_index['sample_id'].nunique()}")
print("\n  Class breakdown:")
for cls, count in master_index["class"].value_counts().items():
    label = 1 if cls in FERNING_CLASSES else 0
    print(f"    {cls} (stage1_label={label}): {count:,} patches")
print("\n  Stage 1 binary split:")
print(f"    Ferning  (1): {(master_index['label_stage1']==1).sum():,}")
print(f"    No Fern  (0): {(master_index['label_stage1']==0).sum():,}")


GENERATING master_patch_index.csv
  Detected Layout B: 7 sample folders
  Sample1_CF: 809 patches (class=CF)
  Sample1_NF: 415 patches (class=NF)
  Sample1_PF: 201 patches (class=PF)
  Sample2_NF: 420 patches (class=NF)
  Sample2_PF: 201 patches (class=PF)
  Sample3_PF: 199 patches (class=PF)
  Sample4_PF: 201 patches (class=PF)

  Saved -> /content/drive/MyDrive/ferning_project/outputs/stage1/master_patch_index.csv

  Total patches  : 2,446
  Unique samples : 7

  Class breakdown:
    NF (stage1_label=0): 835 patches
    CF (stage1_label=1): 809 patches
    PF (stage1_label=1): 802 patches

  Stage 1 binary split:
    Ferning  (1): 1,611
    No Fern  (0): 835



---
## Cell 4 — Generate LOSO Fold Splits
Leave-One-Subject-Out cross-validation: with only 7 source images, each image
takes one turn as the sole held-out validation set (§3.7.3).

`fold_splits.csv` is saved to the shared `outputs/stage1/` directory and
**reused by Stage 2 and PINN notebooks** to guarantee identical splits.

Set `FORCE_REGENERATE_FOLDS = True` to rebuild.


In [8]:

FORCE_REGENERATE_FOLDS = True  # Set False once splits are built

def generate_fold_splits_loso(master_index: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    """
    Leave-One-Sample-Out splits (§3.7.3).
    Each unique sample_id is held out once; all others form the training set.
    Returns a DataFrame with columns: sample_id, fold, split.
    """
    sample_ids = sorted(master_index["sample_id"].unique())
    records = []
    for fold_num, val_id in enumerate(sample_ids, start=1):
        for sid in sample_ids:
            records.append({
                "sample_id": sid,
                "fold"     : fold_num,
                "split"    : "val" if sid == val_id else "train",
            })
    df = pd.DataFrame(records)
    df.to_csv(output_path, index=False)
    print(f"  Saved -> {output_path}")
    return df


print("=" * 60)
print("GENERATING fold_splits.csv  [LOSO]")
print("=" * 60)

if FOLD_SPLITS_PATH.exists() and not FORCE_REGENERATE_FOLDS:
    print("  [SKIP] Already exists — loading from disk.")
    fold_splits = pd.read_csv(FOLD_SPLITS_PATH)
else:
    fold_splits = generate_fold_splits_loso(master_index, FOLD_SPLITS_PATH)

N_FOLDS = fold_splits["fold"].nunique()
print(f"  N_FOLDS = {N_FOLDS}  (one per unique sample)")

# ── Fold summary and validation ────────────────────────────────────────────
print(f"\n  {'Fold':<6} {'Val Sample':<22} {'Val Class':<10} {'Val Patches':>12} {'Train Patches':>14}  Status")
print(f"  {'-'*75}")

all_ok = True
for fold_num in range(1, N_FOLDS + 1):
    fold_data  = fold_splits[fold_splits["fold"] == fold_num]
    val_id     = fold_data[fold_data["split"] == "val"]["sample_id"].iloc[0]
    train_ids  = fold_data[fold_data["split"] == "train"]["sample_id"].tolist()
    val_df_    = master_index[master_index["sample_id"] == val_id]
    train_df_  = master_index[master_index["sample_id"].isin(train_ids)]
    val_cls    = val_df_["class"].iloc[0]
    n_val      = len(val_df_)
    n_train    = len(train_df_)
    train_lbls = train_df_["label_stage1"].unique().tolist()
    ok         = (0 in train_lbls and 1 in train_lbls)
    status     = "[OK]" if ok else "[FATAL] Missing class in train set"
    if not ok:
        all_ok = False
    print(f"  {fold_num:<6} {val_id:<22} {val_cls:<10} {n_val:>12,} {n_train:>14,}  {status}")

print()
if not all_ok:
    raise RuntimeError("One or more folds is missing a class in the training set. Cannot proceed.")
print("[OK] All folds have both classes in training — ready to train.")


GENERATING fold_splits.csv  [LOSO]
  Saved -> /content/drive/MyDrive/ferning_project/outputs/stage1/fold_splits.csv
  N_FOLDS = 7  (one per unique sample)

  Fold   Val Sample             Val Class   Val Patches  Train Patches  Status
  ---------------------------------------------------------------------------
  1      Sample1_CF             CF                  809          1,637  [OK]
  2      Sample1_NF             NF                  415          2,031  [OK]
  3      Sample1_PF             PF                  201          2,245  [OK]
  4      Sample2_NF             NF                  420          2,026  [OK]
  5      Sample2_PF             PF                  201          2,245  [OK]
  6      Sample3_PF             PF                  199          2,247  [OK]
  7      Sample4_PF             PF                  201          2,245  [OK]

[OK] All folds have both classes in training — ready to train.



---
## Cell 5 — Data Verification
Four integrity checks before training begins. Fix any failures before continuing.


In [9]:

print("=" * 60)
print("DATA VERIFICATION")
print("=" * 60)

checks_passed = True

# [1] Required columns
print("\n[1/4] Required columns")
required_cols = ["sample_id", "class", "path", "label_stage1"]
missing_cols  = [c for c in required_cols if c not in master_index.columns]
if missing_cols:
    print(f"  [FAIL] Missing: {missing_cols}")
    checks_passed = False
else:
    print("  [OK] All required columns present")

# [2] File existence
print("\n[2/4] .npy file existence")
all_paths  = master_index["path"].tolist()
n_existing = sum(1 for p in all_paths if os.path.exists(p))
n_missing  = len(all_paths) - n_existing
print(f"  Total : {len(all_paths):,}")
print(f"  Found : {n_existing:,}")
if n_missing > 0:
    print(f"  [WARN] Missing : {n_missing:,} — these will be skipped")
    master_index = master_index[master_index["path"].apply(os.path.exists)].copy()
    if len(master_index) == 0:
        checks_passed = False
        print("  [FAIL] No files found at all — check your DATA_ROOT_OVERRIDE path")
else:
    print("  [OK] All files found")

# [3] .npy format spot-check
print("\n[3/4] .npy format spot-check")
existing_paths = master_index["path"].tolist()
if existing_paths:
    sample = np.load(existing_paths[0])
    print(f"  Shape : {sample.shape}")
    print(f"  dtype : {sample.dtype}")
    print(f"  Range : [{sample.min():.3f}, {sample.max():.3f}]")
    if sample.shape[:2] != INPUT_SHAPE[:2]:
        print(f"  [WARN] Shape {sample.shape[:2]} != expected {INPUT_SHAPE[:2]} — preprocess_npy will resize")
else:
    print("  [FAIL] No files to check")
    checks_passed = False

# [4] LOSO leakage check
print("\n[4/4] LOSO leakage check")
leakage_found = False
for fold_num in range(1, N_FOLDS + 1):
    fold_data = fold_splits[fold_splits["fold"] == fold_num]
    train_ids = set(fold_data[fold_data["split"] == "train"]["sample_id"])
    val_ids   = set(fold_data[fold_data["split"] == "val"]["sample_id"])
    overlap   = train_ids & val_ids
    if overlap:
        print(f"  [FAIL] Fold {fold_num}: overlap detected — {overlap}")
        leakage_found = True
if leakage_found:
    checks_passed = False
    print("  Re-run Cell 4 with FORCE_REGENERATE_FOLDS=True to fix.")
else:
    print(f"  [OK] No leakage detected across {N_FOLDS} folds")

print("\n" + "=" * 60)
if checks_passed:
    print("[OK] ALL CHECKS PASSED — ready to train.")
else:
    print("[FAIL] Fix the issues above before running the training loop.")
    raise RuntimeError("Data verification failed.")
print("=" * 60)


DATA VERIFICATION

[1/4] Required columns
  [OK] All required columns present

[2/4] .npy file existence
  Total : 2,446
  Found : 2,446
  [OK] All files found

[3/4] .npy format spot-check
  Shape : (128, 128, 3)
  dtype : float32
  Range : [-2.084, 2.396]

[4/4] LOSO leakage check
  [OK] No leakage detected across 7 folds

[OK] ALL CHECKS PASSED — ready to train.



---
## Cell 6 — Data Loading & Preprocessing Utilities
Patches are already preprocessed by the preprocessing notebook.
`preprocess_npy()` loads each `.npy` file and ensures correct shape and dtype.


In [10]:

def load_fold_data(fold_num: int):
    """
    Return (train_df, val_df) for the given LOSO fold.
    Patient-level split guarantees no data leakage (§3.7.3).
    """
    fold_data = fold_splits[fold_splits["fold"] == fold_num]
    train_ids = fold_data[fold_data["split"] == "train"]["sample_id"].tolist()
    val_ids   = fold_data[fold_data["split"] == "val"]["sample_id"].tolist()
    train_df  = master_index[master_index["sample_id"].isin(train_ids)].copy()
    val_df    = master_index[master_index["sample_id"].isin(val_ids)].copy()
    return train_df, val_df


def preprocess_npy(npy_path: str) -> np.ndarray:
    """
    Load a preprocessed .npy patch.
    Ensures 3-channel float32 output at INPUT_SHAPE.
    All preprocessing (CLAHE, Gaussian blur, normalisation) was applied
    upstream in the preprocessing notebook.
    """
    img = np.load(npy_path).astype(np.float32)

    # Ensure 3-channel
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    elif img.ndim == 3 and img.shape[-1] == 1:
        img = np.repeat(img, 3, axis=-1)
    elif img.ndim == 3 and img.shape[-1] == 4:
        img = img[:, :, :3]

    # Resize only if necessary (preprocessing notebook should have done this)
    if img.shape[:2] != INPUT_SHAPE[:2]:
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_u8 = np.clip((img * std + mean) * 255, 0, 255).astype(np.uint8)
        img_u8 = cv2.resize(img_u8, (INPUT_SHAPE[1], INPUT_SHAPE[0]),
                            interpolation=cv2.INTER_LINEAR)
        img = img_u8.astype(np.float32) / 255.0
        img = (img - mean) / std

    assert img.shape == INPUT_SHAPE, f"Unexpected shape after load: {img.shape}"
    return img


print("[OK] load_fold_data() defined")
print(f"[OK] preprocess_npy() defined — output shape: {INPUT_SHAPE}")


[OK] load_fold_data() defined
[OK] preprocess_npy() defined — output shape: (128, 128, 3)



---
## Cell 7 — Class-Aware Augmentation & Data Generator (§3.5, Table 6)
Augmentation parameters match Table 6 of the manuscript exactly:
- Rotation ±90° (in 90° steps)
- Horizontal / Vertical flip (50% probability each)
- Brightness adjustment ±20%
- Zoom 0.9–1.1×
- Translation ±10% (via padding + crop)
- Elastic deformation α=15, σ=3

Class-aware multiplier: CF patches oversampled at **1.8×** (§3.5, Table 6).


In [12]:

def augment_patch(img: np.ndarray) -> np.ndarray:
    """
    Stochastic augmentations (§3.5, Table 6).
    Operates on ImageNet-normalised float32 arrays.
    All transforms preserve morphological plausibility of ferning structures.
    """
    h, w = img.shape[:2]

    # Rotation ±90° in 90-degree steps
    k = np.random.randint(0, 4)
    if k > 0:
        img = np.rot90(img, k)

    # Horizontal flip (50%)
    if np.random.rand() < 0.5:
        img = np.fliplr(img)

    # Vertical flip (50%)
    if np.random.rand() < 0.5:
        img = np.flipud(img)

    # Brightness ±20%
    delta = np.random.uniform(-0.20, 0.20)
    img   = img + delta

    # Zoom 0.9–1.1×
    zoom_factor = np.random.uniform(0.90, 1.10)
    new_h = int(h * zoom_factor)
    new_w = int(w * zoom_factor)
    if zoom_factor > 1.0:
        pad_h = (new_h - h) // 2
        pad_w = (new_w - w) // 2
        img = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode="reflect")
    else:
        crop_h = (h - new_h) // 2
        crop_w = (w - new_w) // 2
        img = img[crop_h:crop_h + new_h, crop_w:crop_w + new_w]

    # Resize back to INPUT_SHAPE
    img_min, img_max = img.min(), img.max()
    if img_max > img_min:
        img_norm    = ((img - img_min) / (img_max - img_min) * 255).astype(np.uint8)
        img_resized = cv2.resize(img_norm, (INPUT_SHAPE[1], INPUT_SHAPE[0]),
                                 interpolation=cv2.INTER_LINEAR)
        img = img_resized.astype(np.float32) / 255.0 * (img_max - img_min) + img_min
    else:
        img = cv2.resize(img.astype(np.float32),
                         (INPUT_SHAPE[1], INPUT_SHAPE[0]), interpolation=cv2.INTER_LINEAR)

    # Translation ±10% via reflect-pad + crop
    t_h = int(h * 0.10)
    t_w = int(w * 0.10)
    shift_h = np.random.randint(-t_h, t_h + 1)
    shift_w = np.random.randint(-t_w, t_w + 1)
    img = np.roll(img, shift_h, axis=0)
    img = np.roll(img, shift_w, axis=1)

    # Elastic deformation α=15, σ=3 (§3.5, Table 6)
    alpha, sigma = 15, 3
    rng    = np.random.default_rng()
    dx     = cv2.GaussianBlur(
                 (rng.random((h, w)) * 2 - 1).astype(np.float32), (0, 0), sigma
             ) * alpha
    dy     = cv2.GaussianBlur(
                 (rng.random((h, w)) * 2 - 1).astype(np.float32), (0, 0), sigma
             ) * alpha
    x, y   = np.meshgrid(np.arange(w), np.arange(h))
    map_x  = np.clip(x + dx, 0, w - 1).astype(np.float32)
    map_y  = np.clip(y + dy, 0, h - 1).astype(np.float32)
    img    = cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR,
                       borderMode=cv2.BORDER_REFLECT)

    # Guarantee correct output shape
    if img.shape != INPUT_SHAPE:
        img = cv2.resize(img, (INPUT_SHAPE[1], INPUT_SHAPE[0]),
                         interpolation=cv2.INTER_LINEAR)
        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)

    return img.astype(np.float32)


class NumpyDataGenerator(keras.utils.Sequence):
    """
    On-the-fly data generator with class-aware augmentation (§3.5, Table 6).

    Parameters
    ----------
    dataframe : pd.DataFrame  — must have 'path', 'label_stage1', 'class'
    batch_size : int
    shuffle    : bool
    augment    : bool         — apply augmentation (True for training only)
    """

    def __init__(self, dataframe: pd.DataFrame, batch_size: int = 32,
                 shuffle: bool = True, augment: bool = False):
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.augment    = augment

        if augment:
            # Apply class-aware oversampling (§3.5, Table 6)
            parts = []
            for cls in KNOWN_CLASSES:
                cls_df = dataframe[dataframe["class"] == cls].copy()
                if cls_df.empty:
                    continue
                mult      = AUG_MULTIPLIER.get(cls, 1.0)
                n_repeats = int(mult)
                frac      = mult - n_repeats
                repeated  = pd.concat([cls_df] * n_repeats, ignore_index=True)
                if frac > 0:
                    extra = cls_df.sample(
                        frac=frac, random_state=RANDOM_SEED,
                        replace=(frac > 1.0)
                    )
                    repeated = pd.concat([repeated, extra], ignore_index=True)
                parts.append(repeated)
            self.df = pd.concat(parts, ignore_index=True).sample(
                frac=1, random_state=RANDOM_SEED
            ).reset_index(drop=True)
        else:
            self.df = dataframe.reset_index(drop=True)

        self.n = len(self.df)
        self.on_epoch_end()

    def __len__(self) -> int:
        return int(np.ceil(self.n / self.batch_size))

    def __getitem__(self, index: int):
        start = index * self.batch_size
        end   = min(start + self.batch_size, self.n)
        rows  = self.df.iloc[self.indices[start:end]]
        imgs  = []
        for p in rows["path"]:
            img = preprocess_npy(p)
            if self.augment:
                img = augment_patch(img)
            if img.shape != INPUT_SHAPE:
                img = cv2.resize(img, (INPUT_SHAPE[1], INPUT_SHAPE[0]),
                                 interpolation=cv2.INTER_LINEAR)
                if img.ndim == 2:
                    img = np.stack([img] * 3, axis=-1)
            imgs.append(img.astype(np.float32))
        X = np.stack(imgs, axis=0)
        y = keras.utils.to_categorical(rows["label_stage1"].values,
                                       num_classes=NUM_CLASSES)
        return X, y

    def on_epoch_end(self):
        self.indices = np.arange(self.n)
        if self.shuffle:
            np.random.shuffle(self.indices)

    def reset(self):
        self.on_epoch_end()


print("[OK] augment_patch() defined — parameters match §3.5 Table 6")
print("[OK] NumpyDataGenerator defined — class-aware augmentation enabled")


[OK] augment_patch() defined — parameters match §3.5 Table 6
[OK] NumpyDataGenerator defined — class-aware augmentation enabled



---
## Cell 8 — Architecture Builders (§3.2)
Four architectures as specified in §3.2:
- **EfficientNetB3** — compound scaling
- **DenseNet121** — dense connectivity
- **ResNet50** — residual connections (primary baseline)
- **SwinTransformerTiny** — shifted-window attention (custom Keras implementation;
  `keras_cv` only exposes a **VideoSwinT** backbone requiring 5-D input `(batch, 32, 224, 224, 3)`
  for temporal video tasks — incompatible with 2-D 128×128 image patches.
  No standard 2-D Swin-Tiny with ImageNet weights is available through Keras or keras_cv.
  The custom implementation used here is documented as a limitation in §3.4, Table 3.)

**Classification head**: GlobalAveragePooling → Dropout(0.5) → Dense(128, relu) → Dense(num_classes, softmax)

The Dense(128) hidden layer is a standard practice for adapting pretrained feature maps
to small-domain classification tasks and is applied consistently across all four architectures.
All base networks start with **frozen weights** (Stage A initialisation).


In [14]:

def _build_head(base_output, num_classes: int, is_sequence: bool = False):
    """
    Shared classification head applied identically across all four architectures.
    GlobalAveragePooling → Dropout(0.5) → Dense(128, relu) → Dense(num_classes, softmax)

    is_sequence : set True for Swin (1D token sequence) to use GlobalAveragePooling1D.
    """
    if is_sequence:
        x = layers.GlobalAveragePooling1D()(base_output)
    else:
        x = layers.GlobalAveragePooling2D()(base_output)
    x       = layers.Dropout(DROPOUT_RATE)(x)
    x       = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return outputs


def build_efficientnetb3(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES) -> keras.Model:
    """EfficientNetB3 — compound scaling (§3.2, §3.5). ImageNet pretrained."""
    try:
        base = EfficientNetB3(include_top=False, weights="imagenet",
                              input_shape=input_shape)
    except Exception:
        print("  [WARN] ImageNet weights unavailable — initialising randomly.")
        base = EfficientNetB3(include_top=False, weights=None,
                              input_shape=input_shape)
    base.trainable = False  # Frozen for Stage A
    inputs  = layers.Input(shape=input_shape)
    x       = base(inputs, training=False)
    outputs = _build_head(x, num_classes)
    return keras.Model(inputs, outputs, name="EfficientNetB3_stage1")


def build_densenet121(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES) -> keras.Model:
    """DenseNet121 — dense connectivity (§3.2, §3.5). ImageNet pretrained."""
    try:
        base = DenseNet121(include_top=False, weights="imagenet",
                           input_shape=input_shape)
    except Exception:
        print("  [WARN] ImageNet weights unavailable — initialising randomly.")
        base = DenseNet121(include_top=False, weights=None,
                           input_shape=input_shape)
    base.trainable = False
    inputs  = layers.Input(shape=input_shape)
    x       = base(inputs, training=False)
    outputs = _build_head(x, num_classes)
    return keras.Model(inputs, outputs, name="DenseNet121_stage1")


def build_resnet50(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES) -> keras.Model:
    """ResNet50 — residual connections; primary reference baseline (§3.7.1). ImageNet pretrained."""
    try:
        base = ResNet50(include_top=False, weights="imagenet",
                        input_shape=input_shape)
    except Exception:
        print("  [WARN] ImageNet weights unavailable — initialising randomly.")
        base = ResNet50(include_top=False, weights=None,
                        input_shape=input_shape)
    base.trainable = False
    inputs  = layers.Input(shape=input_shape)
    x       = base(inputs, training=False)
    outputs = _build_head(x, num_classes)
    return keras.Model(inputs, outputs, name="ResNet50_stage1")



def build_swin_transformer_tiny(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES) -> keras.Model:
    """
    Swin Transformer-Tiny — shifted-window self-attention (§3.2, §3.5).

    PRETRAINED WEIGHT NOTE:
    keras_cv 0.9.0 only exposes VideoSwinT (5-D input for video sequences).
    There is no standard 2-D Swin-Tiny with ImageNet weights available through
    the Keras or keras_cv APIs for image patches. This is explicitly documented
    as a limitation in §3.4 Table 3 (Transfer Learning Domain Shift row).
    This custom implementation mirrors the published Swin-Tiny architecture:
      - 4×4 patch embedding → 96-dim tokens
      - 2 hierarchical stages with multi-head self-attention (3 and 6 heads)
      - LayerNorm + GELU feed-forward blocks
    Weights are initialised randomly (Glorot uniform), meaning this architecture
    does not benefit from ImageNet pretrained representations.
    """
    def transformer_block(x, num_heads, name_prefix):
        _, seq_len, dim = x.shape
        x_norm = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_ln1")(x)
        attn   = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=max(1, dim // num_heads),
            dropout=0.0,
            name=f"{name_prefix}_mha"
        )(x_norm, x_norm)
        x      = layers.Add(name=f"{name_prefix}_add1")([x, attn])
        x_norm = layers.LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_ln2")(x)
        ff     = layers.Dense(dim * 4, activation="gelu",
                              name=f"{name_prefix}_ff1")(x_norm)
        ff     = layers.Dense(dim, name=f"{name_prefix}_ff2")(ff)
        return layers.Add(name=f"{name_prefix}_add2")([x, ff])

    inputs = layers.Input(shape=input_shape)

    # Patch embedding: 4×4 non-overlapping patches → 96-dim token sequence
    x = layers.Conv2D(96, kernel_size=4, strides=4, padding="valid",
                      name="patch_embed")(inputs)
    _, h, w, c = x.shape
    x = layers.Reshape((h * w, c), name="patch_flatten")(x)

    # Stage 1: 2 transformer blocks, 3 heads (96-dim)
    for i in range(2):
        x = transformer_block(x, num_heads=3, name_prefix=f"s1_b{i}")

    # Patch merging / Stage 2: project to 192-dim, 2 blocks, 6 heads
    x = layers.Dense(192, name="downsample_1")(x)
    for i in range(2):
        x = transformer_block(x, num_heads=6, name_prefix=f"s2_b{i}")

    x = layers.LayerNormalization(epsilon=1e-6, name="final_ln")(x)

    # Use shared head with is_sequence=True (GlobalAveragePooling1D over token dim)
    outputs = _build_head(x, num_classes, is_sequence=True)

    return keras.Model(inputs, outputs, name="SwinTransformerTiny_stage1")


MODEL_BUILDERS = {
    "EfficientNetB3"     : build_efficientnetb3,
    "DenseNet121"        : build_densenet121,
    "ResNet50"           : build_resnet50,
    "SwinTransformerTiny": build_swin_transformer_tiny,
}

print("[OK] Four architecture builders defined (§3.2, §3.5):")
for name in MODEL_BUILDERS:
    print(f"  - {name}")
print()
print("NOTE: SwinTransformerTiny uses a custom implementation without")
print("      pretrained ImageNet weights (see §3.4 — documented limitation).")


[OK] Four architecture builders defined (§3.2, §3.5):
  - EfficientNetB3
  - DenseNet121
  - ResNet50
  - SwinTransformerTiny

NOTE: SwinTransformerTiny uses a custom implementation without
      pretrained ImageNet weights (see §3.4 — documented limitation).



---
## Cell 9 — Focal Loss (Class Imbalance Mitigation)
Chapter 3 specifies class-aware augmentation (§3.5, Table 6) to address class imbalance.
Focal Loss is used here as the training loss to further down-weight well-classified
majority-class patches and focus learning on difficult minority-class examples.
This is consistent with the class-imbalance mitigation strategy described in §3.5.


In [15]:

class FocalLoss(keras.losses.Loss):
    """
    Focal Loss for class imbalance mitigation.
    L_focal = -alpha_t * (1 - p_t)^gamma * log(p_t)

    gamma : focusing parameter — higher values weight hard examples more
    alpha : per-class weights — computed from training label distribution
    """

    def __init__(self, alpha=None, gamma=2.0, name="focal_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.alpha = alpha
        self.gamma = gamma

    def call(self, y_true, y_pred):
        y_pred   = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true_f = tf.cast(y_true, tf.float32)
        ce       = -y_true_f * tf.math.log(y_pred)
        p_t      = tf.reduce_sum(y_true_f * y_pred, axis=-1, keepdims=True)
        focal    = tf.pow(1.0 - p_t, self.gamma)
        loss     = focal * tf.reduce_sum(ce, axis=-1)

        if self.alpha is not None:
            alpha_t = tf.reduce_sum(
                y_true_f * tf.constant(self.alpha, dtype=tf.float32), axis=-1
            )
            loss = alpha_t * loss

        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"alpha": self.alpha, "gamma": self.gamma})
        return cfg


def build_focal_loss(y_train: np.ndarray, gamma: float = 2.0) -> FocalLoss:
    """Compute class-balanced alpha from training labels and return FocalLoss."""
    classes = np.unique(y_train)
    cw_arr  = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    cw_dict = {int(c): float(w) for c, w in zip(classes, cw_arr)}
    alpha   = [cw_dict.get(0, 1.0), cw_dict.get(1, 1.0)]
    print(f"  FocalLoss alpha={alpha}  gamma={gamma}")
    return FocalLoss(alpha=alpha, gamma=gamma)


print("[OK] FocalLoss defined  (gamma=2.0, alpha computed per fold)")


[OK] FocalLoss defined  (gamma=2.0, alpha computed per fold)



---
## Cell 10 — Evaluation Helper (§3.6)
Computes per-fold metrics: Sensitivity, Specificity, Precision, Accuracy, F1-score.

LOSO-aware: val folds are **single-class by design** (one source image = one class).
Metrics that require both classes will be NaN for those folds.
Final performance is computed by **pooling raw TP/TN/FP/FN counts** across all folds
in the Evaluation notebook (§3.7.3).


In [18]:

def evaluate_fold(y_true, y_pred_proba, threshold=THRESHOLD,
                  model_name="", fold_num=0) -> dict:
    """
    Compute per-fold diagnostic metrics (§3.6).
    Returns a dict with TP, TN, FP, FN and derived metrics.
    NaN is returned for metrics undefined on single-class val folds.
    """
    y_true       = np.asarray(y_true)
    y_pred_proba = np.asarray(y_pred_proba)

    y_pred_pos = (y_pred_proba[:, POSITIVE_CLASS_INDEX]
                  if y_pred_proba.ndim > 1 else y_pred_proba)
    y_pred_cls = (y_pred_pos >= threshold).astype(int)

    # Force (2,2) shape — single-class val folds give partial matrices
    cm = confusion_matrix(y_true, y_pred_cls, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    precision   = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    accuracy    = (tp + tn) / len(y_true) if len(y_true) > 0 else np.nan
    f1 = (2 * precision * sensitivity / (precision + sensitivity)
          if (not np.isnan(precision) and not np.isnan(sensitivity)
              and (precision + sensitivity) > 0) else np.nan)

    val_classes = len(np.unique(y_true))
    print(f"\n  {'='*50}")
    print(f"  {model_name} | Fold {fold_num}")
    print(f"  {'='*50}")
    print(f"  Val patches     : {len(y_true)}  (classes present: {val_classes})")
    print(f"  Confusion matrix: TN={tn} FP={fp} | FN={fn} TP={tp}")
    if val_classes < 2:
        print(f"  NOTE: Single-class val fold (expected in LOSO) — some metrics N/A")
    print(f"  Sensitivity     : {sensitivity:.4f}" if not np.isnan(sensitivity) else "  Sensitivity     : N/A")
    print(f"  Specificity     : {specificity:.4f}" if not np.isnan(specificity) else "  Specificity     : N/A")
    print(f"  Precision       : {precision:.4f}"   if not np.isnan(precision)   else "  Precision       : N/A")
    print(f"  Accuracy        : {accuracy:.4f}"    if not np.isnan(accuracy)    else "  Accuracy        : N/A")
    print(f"  F1-score        : {f1:.4f}"          if not np.isnan(f1)          else "  F1-score        : N/A")

    return dict(
        tp=int(tp), tn=int(tn), fp=int(fp), fn=int(fn),
        sensitivity   = float(sensitivity) if not np.isnan(sensitivity) else np.nan,
        specificity   = float(specificity) if not np.isnan(specificity) else np.nan,
        precision     = float(precision)   if not np.isnan(precision)   else np.nan,
        accuracy      = float(accuracy)    if not np.isnan(accuracy)    else np.nan,
        f1_score      = float(f1)          if not np.isnan(f1)          else np.nan,
        val_single_class = int(val_classes < 2),
    )


print("[OK] evaluate_fold() defined (§3.6)")


[OK] evaluate_fold() defined (§3.6)



---
## Cell 11 — Progressive Fine-Tuning Callback (§3.2)
Three-stage progressive unfreezing implemented as a single `model.fit()` call
controlled by a custom `UnfreezeCallback`. This keeps training history continuous
and avoids the overhead of multiple compile/fit cycles.

The callback triggers at specific epoch boundaries:
- **Epoch 0–9** (Stage A): Base frozen, head-only warmup at lr=1e-4
- **Epoch 10** (→ Stage B): Top 30% of base unfrozen, lr reduced to 1e-5
- **Epoch 30** (→ Stage C): All layers unfrozen, lr reduced to 1e-6

Total max epochs = 10 (A) + 20 (B) + 50 (C) = 80, with EarlyStopping (patience=10)
active from epoch 1 monitoring `val_loss`.


In [19]:

# Total epoch budget for the single fit call
EPOCHS_TOTAL   = EPOCHS_STAGE_A + EPOCHS_STAGE_B + EPOCHS_STAGE_C  # 10+20+50 = 80
STAGE_B_START  = EPOCHS_STAGE_A                                      # epoch 10
STAGE_C_START  = EPOCHS_STAGE_A + EPOCHS_STAGE_B                    # epoch 30


class UnfreezeCallback(keras.callbacks.Callback):
    """
    Implements §3.2 three-stage progressive fine-tuning within a single model.fit().

    Stage A (epochs 0–9)      : Base fully frozen, only head trains (lr=1e-4).
    Stage B (epoch 10 onward) : Top 30% of base unfrozen, lr drops to 1e-5.
    Stage C (epoch 30 onward) : All base layers unfrozen, lr drops to 1e-6.

    After each unfreeze the model is recompiled so the optimiser state
    reflects the new trainable parameter set.
    """

    def __init__(self, y_train, stage_b_start=STAGE_B_START,
                 stage_c_start=STAGE_C_START):
        super().__init__()
        self.y_train       = y_train
        self.stage_b_start = stage_b_start
        self.stage_c_start = stage_c_start
        self._stage        = "A"

    def _get_base(self):
        """Return the backbone sub-model if present (CNN architectures)."""
        for layer in self.model.layers:
            if hasattr(layer, "layers") and len(layer.layers) > 5:
                return layer
        return None  # Swin — flat model, no nested backbone

    def on_epoch_begin(self, epoch, logs=None):
        if epoch == self.stage_b_start and self._stage == "A":
            self._enter_stage_b()
        elif epoch == self.stage_c_start and self._stage == "B":
            self._enter_stage_c()

    def _enter_stage_b(self):
        """Unfreeze top 30% of backbone layers, recompile at lr=1e-5."""
        self._stage = "B"
        base = self._get_base()
        if base is not None:
            n = len(base.layers)
            unfreeze_from = int(n * 0.70)
            for layer in base.layers[:unfreeze_from]:
                layer.trainable = False
            for layer in base.layers[unfreeze_from:]:
                layer.trainable = True
        else:
            # Swin: unfreeze all layers except final Dense output
            for layer in self.model.layers[:-1]:
                layer.trainable = True
        self._recompile(LR_STAGE_B)
        n_train = sum(np.prod(v.shape) for v in self.model.trainable_variables)
        print(f"\n  [Stage B] Unfroze top 30% — trainable params: {n_train:,}  lr=1e-5")

    def _enter_stage_c(self):
        """Unfreeze all layers, recompile at lr=1e-6."""
        self._stage = "C"
        base = self._get_base()
        if base is not None:
            for layer in base.layers:
                layer.trainable = True
        else:
            for layer in self.model.layers:
                layer.trainable = True
        self._recompile(LR_STAGE_C)
        n_train = sum(np.prod(v.shape) for v in self.model.trainable_variables)
        print(f"\n  [Stage C] All layers unfrozen — trainable params: {n_train:,}  lr=1e-6")

    def _recompile(self, lr):
        focal = build_focal_loss(self.y_train)
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss=focal,
            metrics=["accuracy"],
        )


print(f"[OK] UnfreezeCallback defined")
print(f"  Stage A : epochs  0 – {STAGE_B_START-1:>2}  (head only, lr=1e-4)")
print(f"  Stage B : epochs {STAGE_B_START:>2} – {STAGE_C_START-1:>2}  (top 30% unfrozen, lr=1e-5)")
print(f"  Stage C : epochs {STAGE_C_START:>2} – {EPOCHS_TOTAL-1:>2}  (all unfrozen, lr=1e-6, early stop)")
print(f"  Total max epochs: {EPOCHS_TOTAL}")


[OK] UnfreezeCallback defined
  Stage A : epochs  0 –  9  (head only, lr=1e-4)
  Stage B : epochs 10 – 29  (top 30% unfrozen, lr=1e-5)
  Stage C : epochs 30 – 79  (all unfrozen, lr=1e-6, early stop)
  Total max epochs: 80



---
## Cell 12 — Training Loop: All Architectures × All LOSO Folds
For each architecture and each fold:
1. Load fold data and build generators
2. Build model with frozen base (Stage A init)
3. Initial compile at lr=1e-4 (Stage A)
4. Single `model.fit()` for `EPOCHS_TOTAL` epochs — `UnfreezeCallback` handles
   Stage A→B→C transitions automatically at the correct epoch boundaries
5. `EarlyStopping` (patience=10 on val_loss) and `ModelCheckpoint` run throughout
6. Evaluate on held-out val fold
7. Save model weights, val predictions, and training history

Results are written to `all_results.csv` after every fold for crash recovery.
Already-completed folds are automatically skipped on re-run.


In [ ]:
# ── Select which models to train ──────────────────────────────────────────
# Edit this list to run a subset. Default: all four architectures.
SELECTED_MODELS = MODEL_NAMES  # e.g. ["ResNet50"] to train one only

# ── Resume support ─────────────────────────────────────────────────────────
results_csv = STAGE1_OUTPUT_DIR / "all_results.csv"

if results_csv.exists():
    existing_df = pd.read_csv(results_csv)
    all_results = existing_df.to_dict("records")
    print(f"[RESUME] Loaded {len(all_results)} existing result rows.")
    print(f"  Completed: { {m: len([r for r in all_results if r['model']==m]) for m in existing_df['model'].unique()} }")
else:
    all_results = []
    print("[START] No previous results — starting fresh.")

# ── Training loop ──────────────────────────────────────────────────────────
for model_name in SELECTED_MODELS:

    print("\n" + "#" * 70)
    print(f"  MODEL: {model_name}")
    print("#" * 70)

    completed_folds = set(r["fold"] for r in all_results if r["model"] == model_name)

    if completed_folds:
        print(f"  Folds already completed: {sorted(completed_folds)}")

    model_out_dir = STAGE1_OUTPUT_DIR / model_name
    model_out_dir.mkdir(parents=True, exist_ok=True)

    for fold_num in range(1, N_FOLDS + 1):

        if fold_num in completed_folds:
            print(f"  [SKIP] Fold {fold_num} — already done.")
            continue

        print("\n" + "=" * 60)
        print(f"  {model_name}  |  LOSO Fold {fold_num} / {N_FOLDS}")
        print("=" * 60)

        fold_out_dir = model_out_dir / f"fold{fold_num}"
        fold_out_dir.mkdir(parents=True, exist_ok=True)

        # ── Load data ──────────────────────────────────────────────────
        train_df, val_df = load_fold_data(fold_num)

        y_train = train_df["label_stage1"].values

        val_sample_id = fold_splits[
            (fold_splits["fold"] == fold_num) &
            (fold_splits["split"] == "val")
        ]["sample_id"].iloc[0]

        print(f"  Val sample      : {val_sample_id}")
        print(f"  Train patches   : {len(train_df):,}  "
              f"(NF={int((y_train==0).sum())}  Ferning={int((y_train==1).sum())})")
        print(f"  Val patches     : {len(val_df):,}")

        if len(np.unique(y_train)) < 2:
            print(f"  [SKIP] Train set is single-class — cannot train this fold.")
            continue

        # ── Class weights ──────────────────────────────────────────────
        classes = np.unique(y_train)

        cw_arr = compute_class_weight(
            "balanced",
            classes=classes,
            y=y_train
        )

        class_weights = {
            int(c): float(w)
            for c, w in zip(classes, cw_arr)
        }

        print(f"  Class weights   : {class_weights}")

        # ── Generators ────────────────────────────────────────────────
        train_gen = NumpyDataGenerator(
            train_df,
            batch_size=BATCH_SIZE,
            shuffle=True,
            augment=True
        )

        val_gen = NumpyDataGenerator(
            val_df,
            batch_size=BATCH_SIZE,
            shuffle=False,
            augment=False
        )

        print(f"  Train batches   : {len(train_gen)}  Val batches: {len(val_gen)}")

        # ── Build model (base frozen — Stage A) ───────────────────────
        tf.keras.backend.clear_session()

        np.random.seed(RANDOM_SEED)
        tf.random.set_seed(RANDOM_SEED)

        model = MODEL_BUILDERS[model_name](INPUT_SHAPE, NUM_CLASSES)

        n_total = model.count_params()

        n_trainable = sum(
            np.prod(v.shape)
            for v in model.trainable_variables
        )

        print(f"  Total params    : {n_total:,}")
        print(f"  Trainable (head): {n_trainable:,}  [Stage A — base frozen]")

        # ── Initial compile — Stage A lr (head only) ───────────────────
        focal_loss_init = build_focal_loss(y_train)

        model.compile(
            optimizer=keras.optimizers.Adam(
                learning_rate=LR_STAGE_A
            ),
            loss=focal_loss_init,
            metrics=["accuracy"],
        )

        # Prevent TF graph rebuild crash when callback recompiles model
        model.run_eagerly = False
        model.make_train_function()

        # ── Callbacks ─────────────────────────────────────────────────
        callbacks = [

            UnfreezeCallback(y_train=y_train),

            keras.callbacks.ModelCheckpoint(
                filepath=str(fold_out_dir / "best_model.keras"),
                monitor="val_loss",
                mode="min",
                save_best_only=True,
                verbose=0,
            ),

            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=EARLY_STOP_PAT,
                mode="min",
                restore_best_weights=True,
                verbose=1,
            ),

            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=LR_REDUCE_FAC,
                patience=LR_REDUCE_PAT,
                min_lr=LR_MIN,
                mode="min",
                verbose=0,
            ),

            keras.callbacks.CSVLogger(
                str(fold_out_dir / "history.csv"),
                append=False
            ),
        ]

        # ── Single model.fit() — UnfreezeCallback manages A→B→C ───────
        start_time = datetime.now()

        model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS_TOTAL,
            callbacks=callbacks,
            class_weight=class_weights,
            verbose=1,
        )

        elapsed = (datetime.now() - start_time).total_seconds()

        print(f"  Training time   : {elapsed:.0f}s")

        # ── Evaluate on val fold ───────────────────────────────────────
        val_gen.reset()

        y_pred_proba = model.predict(val_gen, verbose=0)

        y_true = val_df["label_stage1"].values

        metrics = evaluate_fold(
            y_true,
            y_pred_proba,
            model_name=model_name,
            fold_num=fold_num
        )

        # ── Save val predictions (y_true + y_score) ───────────────────
        y_score = (
            y_pred_proba[:, POSITIVE_CLASS_INDEX]
            if y_pred_proba.ndim > 1
            else y_pred_proba.ravel()
        )

        np.savez(
            str(fold_out_dir / "val_predictions.npz"),
            y_true=y_true,
            y_score=y_score,
            val_sample=np.array([val_sample_id]),
        )

        print(f"  Saved: val_predictions.npz  ({len(y_true)} patches)")

        # ── Attach metadata and persist ────────────────────────────────
        metrics["model"] = model_name
        metrics["fold"] = fold_num
        metrics["val_sample"] = val_sample_id
        metrics["train_patches"] = int(len(train_df))
        metrics["val_patches"] = int(len(val_df))
        metrics["train_time_s"] = int(elapsed)

        all_results.append(metrics)

        pd.DataFrame(all_results).to_csv(
            results_csv,
            index=False
        )

        print(f"  Saved: all_results.csv  ({len(all_results)} rows total)")

        del model
        tf.keras.backend.clear_session()

print("\n" + "=" * 60)
print("TRAINING SESSION COMPLETE")

remaining = [
    m for m in SELECTED_MODELS
    if len([r for r in all_results if r["model"] == m]) < N_FOLDS
]

if remaining:
    print(f"  Still pending  : {remaining}")
    print("  Re-run this cell — completed folds will be skipped automatically.")
else:
    print(f"  All {len(SELECTED_MODELS)} model(s) fully trained across {N_FOLDS} folds.")
    print("  Proceed to Cell 13 to identify the best architecture.")

print("=" * 60)

[START] No previous results — starting fresh.

######################################################################
  MODEL: EfficientNetB3
######################################################################

  EfficientNetB3  |  LOSO Fold 1 / 7
  Val sample      : Sample1_CF
  Train patches   : 1,637  (NF=835  Ferning=802)
  Val patches     : 809
  Class weights   : {0: 0.9802395209580839, 1: 1.020573566084788}
  Train batches   : 52  Val batches: 26
  Total params    : 10,980,529
  Trainable (head): 196,994  [Stage A — base frozen]
  FocalLoss alpha=[0.9802395209580839, 1.020573566084788]  gamma=2.0
Epoch 1/80
52/52 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.7191 - loss: 0.1157 - val_accuracy: 0.9802 - val_loss: 0.0119 - learning_rate: 1.0000e-04
Epoch 2/80
52/52 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.8295 - loss: 0.0525 - val_accuracy: 0.9543 - val_loss: 0.0145 - learning_rate: 1.0000e-04
Epoch 3/80
52/52 ━━━━━━━━━━━━━━━━━━━━ 73s 1s/step - accuracy: 0.8460 - loss: 


---
## Cell 13 — Best Model Selection & Pipeline Handoff (§3.7.3)
Selects the best architecture by highest pooled sensitivity (§3.7.3).
F1-score is the tiebreaker criterion.

Writes `best_model_name.txt` — the PINN notebook reads this file to know
which architecture to load for Physics-Informed fine-tuning.


In [ ]:

results_df = pd.DataFrame(all_results)
trained    = results_df["model"].unique().tolist()

print("=" * 60)
print("POOLED LOSO RESULTS — Stage 1 Summary")
print("=" * 60)
print(f"\n  {'Model':<22} {'Sens':>8} {'Spec':>8} {'Prec':>8} {'F1':>8} {'Acc':>8}  {'TP':>5} {'FN':>5} {'TN':>5} {'FP':>5}")
print(f"  {'-'*90}")

summary_rows = []
for mn in MODEL_NAMES:
    mdf = results_df[results_df["model"] == mn]
    if mdf.empty:
        continue
    tp = mdf["tp"].sum(); fn = mdf["fn"].sum()
    tn = mdf["tn"].sum(); fp = mdf["fp"].sum()
    total = tp + tn + fp + fn
    sens = tp/(tp+fn)          if (tp+fn)>0 else 0.0
    spec = tn/(tn+fp)          if (tn+fp)>0 else 0.0
    prec = tp/(tp+fp)          if (tp+fp)>0 else 0.0
    acc  = (tp+tn)/total       if total>0    else 0.0
    f1   = (2*prec*sens/(prec+sens) if (prec+sens)>0 else 0.0)
    print(f"  {mn:<22} {sens:>8.4f} {spec:>8.4f} {prec:>8.4f} {f1:>8.4f} {acc:>8.4f}  {tp:>5} {fn:>5} {tn:>5} {fp:>5}")
    summary_rows.append(dict(
        model=mn, pooled_sensitivity=sens, pooled_specificity=spec,
        pooled_precision=prec, pooled_f1=f1, pooled_accuracy=acc,
        tp=int(tp), tn=int(tn), fp=int(fp), fn=int(fn),
    ))

summary_df = pd.DataFrame(summary_rows)

# ── Select best model: highest sensitivity, F1 as tiebreaker (§3.7.3) ────
if not summary_df.empty:
    best_idx   = summary_df.sort_values(
        ["pooled_sensitivity", "pooled_f1"], ascending=False
    ).index[0]
    best_model = summary_df.loc[best_idx, "model"]
    best_sens  = summary_df.loc[best_idx, "pooled_sensitivity"]
    best_f1    = summary_df.loc[best_idx, "pooled_f1"]

    print(f"\n{'='*60}")
    print(f"  Best architecture : {best_model}")
    print(f"  Pooled Sensitivity: {best_sens:.4f}  (primary criterion, §3.7.3)")
    print(f"  Pooled F1-score   : {best_f1:.4f}  (tiebreaker criterion, §3.7.3)")
    print(f"  -> Proceeds to Phase 4 PINN fine-tuning (§3.2 Phase 4)")
    print(f"{'='*60}")

    # ── Write handoff files ────────────────────────────────────────────
    (STAGE1_OUTPUT_DIR / "best_model_name.txt").write_text(best_model)
    summary_df.to_csv(STAGE1_OUTPUT_DIR / "model_comparison_summary.csv", index=False)
    results_df.to_csv(STAGE1_OUTPUT_DIR / "all_results.csv", index=False)

    print(f"\n  Handoff files written to: {STAGE1_OUTPUT_DIR}")
    print(f"  best_model_name.txt        -> PINN notebook reads this")
    print(f"  model_comparison_summary.csv -> Evaluation notebook reads this")
    print(f"  all_results.csv              -> Evaluation notebook reads this")
    print(f"  fold_splits.csv              -> Stage 2 & PINN read this")
    print(f"  master_patch_index.csv       -> Stage 2 reads this")

    print(f"\n  Per-model fold weights saved at:")
    for mn in trained:
        for fn in range(1, N_FOLDS + 1):
            mp = STAGE1_OUTPUT_DIR / mn / f"fold{fn}" / "best_model.keras"
            status = "[EXISTS]" if mp.exists() else "[MISSING]"
            print(f"    {mn}/fold{fn}/best_model.keras  {status}")
else:
    print("[WARN] No results found. Run Cell 12 first.")



---
## Cell 14 — Stage 1 Complete
All outputs are saved. Proceed to the next notebook in the pipeline.


In [ ]:

print("=" * 60)
print("STAGE 1 NOTEBOOK — COMPLETE")
print("=" * 60)
print()
print("Outputs saved to:", STAGE1_OUTPUT_DIR)
print()

output_files = [
    ("master_patch_index.csv",       "Stage 2"),
    ("fold_splits.csv",              "Stage 2, PINN"),
    ("all_results.csv",              "Evaluation"),
    ("model_comparison_summary.csv", "Evaluation"),
    ("best_model_name.txt",          "PINN"),
]
for fname, used_by in output_files:
    fpath  = STAGE1_OUTPUT_DIR / fname
    status = "[EXISTS]" if fpath.exists() else "[MISSING]"
    print(f"  {status}  {fname:<35}  -> {used_by}")

print()
print("Per-fold model weights and val_predictions.npz:")
best_model_name = (STAGE1_OUTPUT_DIR / "best_model_name.txt").read_text().strip() \
    if (STAGE1_OUTPUT_DIR / "best_model_name.txt").exists() else "?"
print(f"  Best model: {best_model_name}")
for fn in range(1, N_FOLDS + 1):
    mp = STAGE1_OUTPUT_DIR / best_model_name / f"fold{fn}" / "best_model.keras"
    vp = STAGE1_OUTPUT_DIR / best_model_name / f"fold{fn}" / "val_predictions.npz"
    print(f"  Fold {fn}: model={'[OK]' if mp.exists() else '[MISSING]'}  "
          f"predictions={'[OK]' if vp.exists() else '[MISSING]'}")

print()
print("Next steps:")
print("  1. Run 02_stage2_training.ipynb  (uses fold_splits.csv + master_patch_index.csv)")
print("  2. Run 03_pinn_finetuning.ipynb  (uses best_model_name.txt + best_model.keras)")
print("  3. Run 04_evaluation.ipynb       (uses all_results.csv + val_predictions.npz)")
print("=" * 60)
